In [1]:
import os
import sys
import json
import time
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from scipy.spatial.transform import Rotation
from PIL import Image
import cv2
import trimesh

In [2]:
PROJECT_ROOT = Path(".").resolve().parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from objects_config import OBJECTS, gt_pose_to_matrix, rotation_error_deg, \
                           position_error_cm, add_score

GD_CONFIG  = Path.home() / "rai_workspace/grounding_dino_weights/GroundingDINO_SwinT_OGC.py"
GD_WEIGHTS = Path.home() / "rai_workspace/grounding_dino_weights/groundingdino_swint_ogc.pth"
CAM_INFO   = PROJECT_ROOT / "outputs/camera_info.json"
OUT_DIR    = PROJECT_ROOT / "outputs/ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BOX_THRESHOLD      = 0.25
TEXT_THRESHOLD     = 0.20
MAX_PER_CAMERA     = 2
MAX_TOTAL          = 8
CLUSTER_DIST_M     = 0.30
N_REFINER_ITER     = 5
DEVICE             = "cuda"

In [3]:
def boxes_cxcywh_to_xyxy(boxes_norm, W, H):
    cx, cy, w, h = boxes_norm.unbind(-1)
    return torch.stack([(cx-w/2)*W, (cy-h/2)*H,
                        (cx+w/2)*W, (cy+h/2)*H], dim=-1)


def cluster_estimates(estimates, dist_m=CLUSTER_DIST_M):
    clusters = []
    for est in estimates:
        placed = False
        for cl in clusters:
            if np.linalg.norm(est["position"] - cl["rep"]["position"]) < dist_m:
                cl["members"].append(est)
                if est["score"] > cl["rep"]["score"]:
                    cl["rep"] = est
                placed = True
                break
        if not placed:
            clusters.append({"rep": est, "members": [est]})
    clusters.sort(key=lambda c: c["rep"]["score"], reverse=True)
    return clusters

In [4]:
with open(CAM_INFO) as f:
    camera_info = json.load(f)

In [5]:
print("Loading GroundingDINO …")
from groundingdino.util.inference import load_model, load_image, predict
gd_model = load_model(str(GD_CONFIG), str(GD_WEIGHTS))
gd_model.eval()

Loading GroundingDINO …


final text_encoder_type: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GroundingDINO(
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x DeformableTransformerEncoderLayer(
          (self_attn): MultiScaleDeformableAttention(
            (sampling_offsets): Linear(in_features=256, out_features=256, bias=True)
            (attention_weights): Linear(in_features=256, out_features=128, bias=True)
            (value_proj): Linear(in_features=256, out_features=256, bias=True)
            (output_proj): Linear(in_features=256, out_features=256, bias=True)
          )
          (dropout1): Dropout(p=0.0, inplace=False)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (linear1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout2): Dropout(p=0.0, inplace=False)
          (linear2): Linear(in_features=2048, out_features=256, bias=True)
          (dropout3): Dropout(p=0.0, inplace=False)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_aff

In [6]:
if "HAPPYPOSE_DATA_DIR" not in os.environ:
    os.environ["HAPPYPOSE_DATA_DIR"] = "/home/salman/rai_workspace/happypose_data"  # <-- update if different

print("HAPPYPOSE_DATA_DIR:", os.environ["HAPPYPOSE_DATA_DIR"])

HAPPYPOSE_DATA_DIR: /home/salman/happypose_data


In [7]:
os.environ["HAPPYPOSE_DATA_DIR"] = "/home/salman/rai_workspace/happypose_data"
print(os.environ["HAPPYPOSE_DATA_DIR"])

/home/salman/rai_workspace/happypose_data


In [8]:
print("Loading MegaPose …")
from happypose.toolbox.datasets.object_dataset import RigidObjectDataset, RigidObject
from happypose.toolbox.utils.load_model import load_named_model
from happypose.toolbox.inference.types import ObservationTensor
from happypose.toolbox.utils.tensor_collection import PandasTensorCollection

object_dataset = RigidObjectDataset([
    RigidObject(
        label      = label,
        mesh_path  = PROJECT_ROOT / "megapose_objects" / label / "mesh.ply",
        scaling_factor = 1.0,
    )
    for label in OBJECTS
])
pose_estimator = load_named_model("megapose-1.0-RGBD", object_dataset=object_dataset)
pose_estimator._SO3_grid = pose_estimator._SO3_grid.to(DEVICE)

Loading MegaPose …
MKL_NUM_THREADS: 1
OMP_NUM_THREADS: 1
CUDA_VISIBLE_DEVICES: 0
EGL_VISIBLE_DEVICES: 0


Known pipe types:
  glxGraphicsPipe
(1 aux display modules not yet loaded.)
Known pipe types:
  glxGraphicsPipe
(1 aux display modules not yet loaded.)
Known pipe types:
  glxGraphicsPipe
(1 aux display modules not yet loaded.)
Known pipe types:
  glxGraphicsPipe
(1 aux display modules not yet loaded.)


In [9]:
results = {}

for label, cfg in OBJECTS.items():
    print(f"\n{'='*60}")
    print(f"Object: {label}  prompt: '{cfg['prompt']}'")
    print(f"{'='*60}")

    T_gt   = gt_pose_to_matrix(cfg["gt_pose"])
    result = {"status": "no_detection"}

    t0_det = time.perf_counter()
    all_dets = []

    for cam_name, info in camera_info.items():
        W, H = info["width"], info["height"]
        _, img_tensor = load_image(info["rgb_path"])

        boxes_norm, logits, _ = predict(
            model          = gd_model,
            image          = img_tensor,
            caption        = cfg["prompt"],
            box_threshold  = BOX_THRESHOLD,
            text_threshold = TEXT_THRESHOLD,
        )
        if len(logits) == 0:
            continue

        boxes_xyxy = boxes_cxcywh_to_xyxy(boxes_norm, W, H)
        order      = logits.argsort(descending=True)[:MAX_PER_CAMERA]
        for i in order:
            all_dets.append({
                "score":    logits[i].item(),
                "cam":      cam_name,
                "box_xyxy": boxes_xyxy[i],
                "info":     info,
            })

    det_time = time.perf_counter() - t0_det

    if not all_dets:
        print(f"  No detections — skipping")
        results[label] = result
        continue

    all_dets.sort(key=lambda d: d["score"], reverse=True)
    top_dets = all_dets[:MAX_TOTAL]
    print(f"  {len(top_dets)} detection(s) in {det_time:.2f}s")

    t0_est  = time.perf_counter()
    estimates = []

    for det in top_dets:
        info = det["info"]
        rgb   = np.array(Image.open(info["rgb_path"]).convert("RGB"))
        depth = np.load(info["depth_path"])
        K     = np.array(info["K"], dtype=np.float64)
        T_wc  = np.array(info["T_world_cam"])

        obs = ObservationTensor.from_numpy(rgb=rgb, depth=depth, K=K).to(DEVICE)
        box = det["box_xyxy"].unsqueeze(0).float().to(DEVICE)

        tc = PandasTensorCollection(
            infos=pd.DataFrame({
                "label":       [label],
                "batch_im_id": [0],
                "instance_id": [0],
                "score":       [float(det["score"])],
            }),
            bboxes=box,
        )
        try:
            out, _ = pose_estimator.run_inference_pipeline(
                obs,
                detections         = tc,
                run_detector       = False,
                n_refiner_iterations = N_REFINER_ITER,
                n_pose_hypotheses  = 1,
            )
        except Exception as e:
            print(f"    MegaPose failed: {e}")
            continue

        T_co = out.poses[0].cpu().numpy()
        T_wo = T_wc @ T_co
        estimates.append({
            "position": T_wo[:3, 3],
            "score":    det["score"],
            "T_wo":     T_wo,
        })

    est_time = time.perf_counter() - t0_est

    if not estimates:
        print(f"  No pose estimates produced")
        result["status"] = "estimation_failed"
        results[label]   = result
        continue

    clusters = cluster_estimates(estimates)
    best     = clusters[0]["rep"]
    T_est    = best["T_wo"]

    pos_err  = position_error_cm(T_est[:3, 3], T_gt[:3, 3])
    rot_err  = rotation_error_deg(T_est[:3, :3], T_gt[:3, :3])
    verts    = np.array(trimesh.load(
                   str(PROJECT_ROOT / "megapose_objects" / label / "mesh.ply")
               ).vertices)
    add_cm, diam_cm, add_ok = add_score(verts, T_est, T_gt)

    print(f"  Position error : {pos_err:.1f} cm")
    print(f"  Rotation error : {rot_err:.1f} deg")
    print(f"  ADD            : {add_cm:.3f} cm  (diam={diam_cm:.2f} cm)  {'✓' if add_ok else '✗'}")
    print(f"  Time           : det={det_time:.2f}s  est={est_time:.2f}s")

    results[label] = {
        "status":            "success",
        "position_error_cm": round(pos_err, 3),
        "rotation_error_deg": round(rot_err, 3),
        "add_cm":            round(add_cm, 4),
        "add_diameter_cm":   round(diam_cm, 4),
        "add_success":       bool(add_ok),
        "detection_time_s":  round(det_time, 3),
        "estimation_time_s": round(est_time, 3),
        "total_time_s":      round(det_time + est_time, 3),
        "n_detections_used": len(estimates),
        "T_est":             T_est.tolist(),
        "T_gt":              T_gt.tolist(),
    }


Object: lamp  prompt: 'yellow lamp'


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


  8 detection(s) in 2.17s
  Position error : 138.7 cm
  Rotation error : 151.6 deg
  ADD            : 138.610 cm  (diam=13.56 cm)  ✗
  Time           : det=2.17s  est=33.51s

Object: cone  prompt: 'cone'


  8 detection(s) in 0.99s
  Position error : 192.6 cm
  Rotation error : 81.9 deg
  ADD            : 197.917 cm  (diam=14.99 cm)  ✗
  Time           : det=0.99s  est=19.42s

Object: redcup  prompt: 'red cup'


  8 detection(s) in 0.90s
  Position error : 104.7 cm
  Rotation error : 98.1 deg
  ADD            : 105.638 cm  (diam=13.69 cm)  ✗
  Time           : det=0.90s  est=39.98s

Object: bottle  prompt: 'bottle'


  8 detection(s) in 0.87s
  Position error : 41.0 cm
  Rotation error : 149.1 deg
  ADD            : 42.974 cm  (diam=27.36 cm)  ✗
  Time           : det=0.87s  est=35.52s

Object: crackerbox  prompt: 'cracker box'


  8 detection(s) in 0.96s
  Position error : 463.7 cm
  Rotation error : 179.3 deg
  ADD            : 452.115 cm  (diam=17.01 cm)  ✗
  Time           : det=0.96s  est=271.11s

Object: deodorant  prompt: 'deodorant'


  8 detection(s) in 0.98s
  Position error : 108.1 cm
  Rotation error : 139.1 deg
  ADD            : 109.970 cm  (diam=17.82 cm)  ✗
  Time           : det=0.98s  est=36.85s


In [ ]:
output = {
    "method":  "megapose",
    "results": results,
}
out_path = OUT_DIR / "results_megapose.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"\nSaved → {out_path}")


Saved → /home/salman/rai_workspace/ablation_study/outputs/ablation/results_megapose.json


: 